Phase 8 -- Ablation studies.
 
1. k sensitivity: does the choice of k in k-NN graph construction
   materially affect GAT performance?
2. Graph-vs-no-graph: with the SAME GAT architecture, does having real
   k-NN relational edges outperform a degenerate graph with no edges
   at all (each node only attends to itself)? This isolates the
   contribution of graph structure from the contribution of simply
   having more model parameters than a linear baseline.
 
Both ablations reuse the EXACT SAME train/val/test node split from
Phase 4 (data.train_mask etc.) so results are directly comparable --
only the edge structure changes between runs.

In [1]:
import importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

In [2]:
_spec = importlib.util.spec_from_file_location("models05", "05_models.py")
_models05 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_models05)
GAT = _models05.GAT

Loaded graph: Data(x=[704, 42], edge_index=[2, 10048], y=[704], train_mask=[704], val_mask=[704], test_mask=[704])

MODEL ARCHITECTURES
GCN(
  (conv1): GCNConv(42, 16)
  (conv2): GCNConv(16, 2)
)
GCN trainable parameters: 722

GAT(
  (conv1): GATConv(42, 16, heads=8)
  (conv2): GATConv(128, 2, heads=1)
)
GAT trainable parameters: 6,022

FORWARD PASS SANITY CHECK (untrained weights, just checking shapes)
GCN output shape: torch.Size([704, 2]) (expected [704, 2])
GAT output shape: torch.Size([704, 2]) (expected [704, 2])

Both models produce correctly shaped output. Ready for Phase 6 (training).


In [3]:
X_PATH = "Data/X_features.csv"
Y_PATH = "Data/y_labels.csv"
DATA_PATH = "Output/graph_data.pt"
OUT_DIR = "Output"

In [4]:
torch.manual_seed(42)
np.random.seed(42)
 
X = pd.read_csv(X_PATH).values.astype(np.float32)
y = pd.read_csv(Y_PATH).values.ravel().astype(np.int64)
base_data = torch.load(DATA_PATH, weights_only=False)  # reuse its train/val/test masks
 
train_labels = torch.tensor(y)[base_data.train_mask]
class_counts = torch.bincount(train_labels)
class_weights = 1.0 / class_counts.float()
class_weights = class_weights / class_weights.sum() * len(class_counts)

In [5]:
def build_knn_edge_index(X, k):
    """Same logic as Phase 3, self-loop-safe, returned as a torch edge_index."""
    nbrs = NearestNeighbors(n_neighbors=k + 1, metric="euclidean").fit(X)
    _, indices = nbrs.kneighbors(X)
    edges = set()
    for node_idx in range(X.shape[0]):
        neighbors_no_self = [n for n in indices[node_idx] if n != node_idx][:k]
        for neighbor_idx in neighbors_no_self:
            edges.add(tuple(sorted((node_idx, int(neighbor_idx)))))
    edge_list = list(edges)
    src = [e[0] for e in edge_list] + [e[1] for e in edge_list]
    dst = [e[1] for e in edge_list] + [e[0] for e in edge_list]
    return torch.tensor([src, dst], dtype=torch.long)
 
 
def quick_train_eval(x, edge_index, y_tensor, train_mask, val_mask, test_mask,
                      max_epochs=150, patience=20, seed=42):
    """Shortened training loop (same logic as Phase 6) for ablation speed."""
    torch.manual_seed(seed)
    model = GAT(in_channels=x.shape[1], hidden_channels=16, out_channels=2, heads=8, dropout=0.6)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
 
    best_val_f1, best_state, no_improve = -1, None, 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(x, edge_index)
        loss = criterion(out[train_mask], y_tensor[train_mask])
        loss.backward()
        optimizer.step()
 
        model.eval()
        with torch.no_grad():
            out_val = model(x, edge_index)
        val_pred = out_val[val_mask].argmax(dim=1)
        val_f1 = f1_score(y_tensor[val_mask].numpy(), val_pred.numpy(), zero_division=0)
 
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
 
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out_test = model(x, edge_index)
        probs_test = torch.softmax(out_test, dim=1)[:, 1]
    preds_test = out_test[test_mask].argmax(dim=1).numpy()
    y_test = y_tensor[test_mask].numpy()
    prob_test = probs_test[test_mask].numpy()
 
    return {
        "accuracy": accuracy_score(y_test, preds_test),
        "precision": precision_score(y_test, preds_test, zero_division=0),
        "recall": recall_score(y_test, preds_test, zero_division=0),
        "f1": f1_score(y_test, preds_test, zero_division=0),
        "auc": roc_auc_score(y_test, prob_test),
    }

In [6]:
x_tensor = torch.tensor(X, dtype=torch.float)
y_tensor = torch.tensor(y, dtype=torch.long)
train_mask, val_mask, test_mask = base_data.train_mask, base_data.val_mask, base_data.test_mask

Ablation 1: k sensitivity

In [7]:
print("=" * 70)
print("ABLATION 1: k SENSITIVITY (same train/val/test split every run)")
print("=" * 70)
k_results = {}
for k in [3, 5, 10, 15, 20]:
    edge_index_k = build_knn_edge_index(X, k)
    metrics = quick_train_eval(x_tensor, edge_index_k, y_tensor, train_mask, val_mask, test_mask)
    k_results[k] = metrics
    print(f"k={k:3d} | edges={edge_index_k.shape[1]//2:5d} | "
          f"F1={metrics['f1']:.4f} | Precision={metrics['precision']:.4f} | "
          f"Recall={metrics['recall']:.4f} | AUC={metrics['auc']:.4f}")

ABLATION 1: k SENSITIVITY (same train/val/test split every run)
k=  3 | edges= 1586 | F1=0.8667 | Precision=0.8125 | Recall=0.9286 | AUC=0.9766
k=  5 | edges= 2576 | F1=0.8276 | Precision=0.8000 | Recall=0.8571 | AUC=0.9702
k= 10 | edges= 5023 | F1=0.8621 | Precision=0.8333 | Recall=0.8929 | AUC=0.9739
k= 15 | edges= 7427 | F1=0.8475 | Precision=0.8065 | Recall=0.8929 | AUC=0.9739
k= 20 | edges= 9803 | F1=0.8929 | Precision=0.8929 | Recall=0.8929 | AUC=0.9794


Ablation 2: graph vs. no-graph (same architecture, zero edges)

In [8]:
print("\n" + "=" * 70)
print("ABLATION 2: DOES THE GRAPH STRUCTURE ITSELF HELP?")
print("=" * 70)
print("Comparing k=10 real graph vs. an EMPTY edge_index (each node only")
print("attends to itself via GATConv's internal self-loop) -- same model,")
print("same parameter count, same training protocol. Only the relational")
print("structure differs.")
 
empty_edge_index = torch.empty((2, 0), dtype=torch.long)
no_graph_metrics = quick_train_eval(x_tensor, empty_edge_index, y_tensor,
                                     train_mask, val_mask, test_mask)
with_graph_metrics = k_results[10]  # reuse the k=10 run from ablation 1
 
print(f"\nNo graph (self only): F1={no_graph_metrics['f1']:.4f}, "
      f"Precision={no_graph_metrics['precision']:.4f}, Recall={no_graph_metrics['recall']:.4f}, "
      f"AUC={no_graph_metrics['auc']:.4f}")
print(f"With k=10 graph:       F1={with_graph_metrics['f1']:.4f}, "
      f"Precision={with_graph_metrics['precision']:.4f}, Recall={with_graph_metrics['recall']:.4f}, "
      f"AUC={with_graph_metrics['auc']:.4f}")
 
f1_diff = with_graph_metrics["f1"] - no_graph_metrics["f1"]
print(f"\nF1 difference (with graph - without): {f1_diff:+.4f}")
if f1_diff > 0.01:
    print("--> The k-NN relational structure provides a measurable benefit "
          "over the same architecture with no neighbor information.")
elif f1_diff < -0.01:
    print("--> Surprisingly, removing the graph structure performed BETTER "
          "here -- on this dataset/split, neighbor aggregation may be "
          "adding noise rather than signal. Worth reporting honestly.")
else:
    print("--> Negligible difference -- on this dataset, most of the GAT's "
          "predictive power comes from the per-node feature transform, "
          "not the relational aggregation. Consistent with the linear-"
          "separability finding from Phase 6.")


ABLATION 2: DOES THE GRAPH STRUCTURE ITSELF HELP?
Comparing k=10 real graph vs. an EMPTY edge_index (each node only
attends to itself via GATConv's internal self-loop) -- same model,
same parameter count, same training protocol. Only the relational
structure differs.

No graph (self only): F1=0.7222, Precision=0.5909, Recall=0.9286, AUC=0.9528
With k=10 graph:       F1=0.8621, Precision=0.8333, Recall=0.8929, AUC=0.9739

F1 difference (with graph - without): +0.1398
--> The k-NN relational structure provides a measurable benefit over the same architecture with no neighbor information.


Save ablation results + plot

In [9]:
ablation_df = pd.DataFrame(k_results).T
ablation_df.index.name = "k"
ablation_df.to_csv(f"{OUT_DIR}/ablation_k_sensitivity.csv")
 
no_graph_df = pd.DataFrame([no_graph_metrics], index=["no_graph"])
no_graph_df.to_csv(f"{OUT_DIR}/ablation_no_graph.csv")
 
fig, ax = plt.subplots(figsize=(7, 4.5))
ks = list(k_results.keys())
f1s = [k_results[k]["f1"] for k in ks]
ax.plot(ks, f1s, marker="o", color="#DD8452", label="GAT with k-NN graph")
ax.axhline(no_graph_metrics["f1"], color="#4C72B0", linestyle="--",
           label=f"GAT, no graph (F1={no_graph_metrics['f1']:.3f})")
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("Test F1")
ax.set_title("Ablation: effect of k and graph structure on GAT performance")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ablation_plot.png", dpi=120)
plt.close()
 
print(f"\nSaved ablation results to {OUT_DIR}/ablation_k_sensitivity.csv, "
      f"{OUT_DIR}/ablation_no_graph.csv, and {OUT_DIR}/ablation_plot.png")


Saved ablation results to Output/ablation_k_sensitivity.csv, Output/ablation_no_graph.csv, and Output/ablation_plot.png
